Note: to run this noteboook follow the instructions in the ```setup-old.sh``` file and activate the MolFLAE2 environment

#### 1. Testing the standalone encoder on a molecule from the CSD

In [ ]:
import argparse
import torch 

from utils.config import load_config
from utils.data_loading import process_sdf_files_to_list
from model.encoder_standalone_cpu import Encoder, molecule_to_latent

/home/teaching/miniconda3/envs/MolFLAE/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
parser = argparse.ArgumentParser()
parser.add_argument('--sdf_folder', type=str,default='data/latent_experiment/val')
parser.add_argument('--output_folder', type=str,default='latent_experiment/ex1/output')
parser.add_argument('--ckpt_path', type=str,default='ckpt-zinc9M/model-epoch=24-val_loss=3.40.ckpt')
parser.add_argument('--config', type=str, default='config.yaml')
parser.add_argument('--batch_size', type=int, default=100)
parser.add_argument('--device', type=str, default='cpu')

args, unknown = parser.parse_known_args()

In [3]:
import torch.nn as nn

# Extract encoder configuration from the config file
cfg = load_config(args.config)

# Create an encoder-only object
encoder = Encoder(**cfg['encoder_config'])
Wh_mu = nn.Linear(
    cfg['encoder_config']['hidden_dim'],
    cfg['optimal_layer_config']['latent_dim']
)
Wh_log_var = nn.Linear(cfg['encoder_config']['hidden_dim'], cfg['optimal_layer_config']['latent_dim'])
Wx_log_var = nn.Linear(cfg['encoder_config']['hidden_dim'], 1) 

encoder.load_state_dict(torch.load("weights/encoder_weights.pth"))
Wh_mu.load_state_dict(torch.load("weights/encoder_weights_KL.pth"))
Wh_log_var.load_state_dict(torch.load("weights/encoder_weights_KL_Wh_log_var.pth"))
Wx_log_var.load_state_dict(torch.load("weights/encoder_weights_KL_Wx_log_var.pth"))

# Include the KL divergence layers into the encoder model
encoder.Wh_mu = Wh_mu
encoder.Wh_log_var = Wh_log_var
encoder.Wx_log_var = Wx_log_var

encoder.eval()
print(encoder)

Encoder:
UniTransformerO2(num_blocks=1, num_layers=9, n_heads=16, act_fn=relu, norm=True, cutoff_mode=global, ew_net_type=r, init h emb: AttentionLayerO2TwoUpdateNodeGeneral(
  (distance_expansion): GaussianSmearing(start=0.0, stop=10.0, num_gaussians=20)
  (x2h_layers): ModuleList(
    (0): BaseX2HAttLayer(
      (hk_func): MLP(
        (net): Sequential(
          (0): Linear(in_features=340, out_features=128, bias=True)
          (1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (2): ReLU()
          (3): Linear(in_features=128, out_features=128, bias=True)
        )
      )
      (hv_func): MLP(
        (net): Sequential(
          (0): Linear(in_features=340, out_features=128, bias=True)
          (1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (2): ReLU()
          (3): Linear(in_features=128, out_features=128, bias=True)
        )
      )
      (hq_func): MLP(
        (net): Sequential(
          (0): Linear(in_features=128, out_features=1

In [4]:
# Load the molecules
mols=process_sdf_files_to_list(args.sdf_folder)

# Build a numbered atom batch
all_h = torch.cat([entry['h'] for entry in mols], dim=0)
all_x = torch.cat([entry['x'] for entry in mols], dim=0)
all_batch = []
current_index = 0
for entry in mols:
    atom_num = entry['atom_num']
    all_batch += [current_index] * atom_num
    current_index += 1
all_batch = torch.tensor(all_batch, dtype=torch.long)

Processing molecules: 100%|██████████| 932/932 [00:00<00:00, 2989.82mol/s]


In [5]:
encoder.eval()
mol0 = mols[0]             # single molecule entry
Zh, Zx, global_batch = molecule_to_latent(encoder, mol0)
print("Zh shape:", Zh.shape)
print("Zx shape:", Zx.shape)


Zh shape: torch.Size([10, 32])
Zx shape: torch.Size([10, 3])


#### 2. Simple tests on the full, charge-enabled VAE (beta)

In [6]:
import torch, math, numpy as np
import argparse

from model.bfn4sbdd import BFN_charge
from config.config import load_config
torch.manual_seed(0)
device = 'cpu'

parser = argparse.ArgumentParser()
parser.add_argument('--sdf_folder', type=str,default='data/latent_experiment/val')
parser.add_argument('--output_folder', type=str,default='latent_experiment/ex1/output')
parser.add_argument('--ckpt_path', type=str,default='ckpt-zinc9M/model-epoch=24-val_loss=3.40.ckpt')
parser.add_argument('--config', type=str, default='config.yaml')
parser.add_argument('--batch_size', type=int, default=100)
parser.add_argument('--device', type=str, default='cpu')

args, unknown = parser.parse_known_args()

config = load_config(args.config)
net_config = config["decoder_config_charge"]["net_config"]

##### 2.1 Generate some toy data and initialise the model

In [7]:
# small toy data
n_protein = 8
n_ligand = 6  # total ligand nodes across batch
K = 5

protein_pos = torch.randn((n_protein, 3), device=device)
protein_v   = torch.randn((n_protein, 27), device=device)
batch_protein = torch.zeros(n_protein, dtype=torch.long, device=device)

# ligand indexing: two molecules with 3 atoms each
batch_ligand = torch.tensor([0,0,0,1,1,1], dtype=torch.long, device=device)

ligand_pos = torch.randn((n_ligand, 3), device=device)
ligand_v   = torch.randint(0, K, (n_ligand,), dtype=torch.long, device=device)
ligand_charges = torch.randn((n_ligand,1), device=device) * 0.1

model = BFN_charge(net_config, protein_atom_feature_dim=27, ligand_atom_feature_dim=K, device=device)
model.eval()


BFN_charge(
  (unio2net): UniTransformerO2(num_blocks=1, num_layers=9, n_heads=16, act_fn=relu, norm=True, cutoff_mode=~global, ew_net_type=global, init h emb: AttentionLayerO2TwoUpdateNodeGeneral(
    (distance_expansion): GaussianSmearing(start=0.0, stop=10.0, num_gaussians=20)
    (x2h_layers): ModuleList(
      (0): BaseX2HAttLayer(
        (hk_func): MLP(
          (net): Sequential(
            (0): Linear(in_features=340, out_features=128, bias=True)
            (1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
            (2): ReLU()
            (3): Linear(in_features=128, out_features=128, bias=True)
          )
        )
        (hv_func): MLP(
          (net): Sequential(
            (0): Linear(in_features=340, out_features=128, bias=True)
            (1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
            (2): ReLU()
            (3): Linear(in_features=128, out_features=128, bias=True)
          )
        )
        (hq_func): MLP(
          (net): 

##### 2.2 Check if the gradients are accumulated in the charge prediction head

In [8]:
model.train()
# ensure model.include_charge True
model.include_charge = True

closs, dloss, qloss = model.loss_one_step(
    t=torch.zeros((n_ligand,1), device=device),
    protein_pos=protein_pos,
    protein_v=protein_v,
    batch_protein=batch_protein,
    ligand_pos=ligand_pos,
    ligand_v=ligand_v,
    batch_ligand=batch_ligand,
    ligand_charges=ligand_charges
)
loss = closs.mean() + dloss.mean() + qloss.mean()
loss.backward()
print("charge_head grads exist?:", any(p.grad is not None for p in model.charge_head.parameters()))


charge_head grads exist?: True


##### 2.3 Verify charge predictions form an untrained model are rotation-invariant

In [9]:
import torch

# make sure final_h is scalar-only or your interdependency_modeling handles vector parts
def random_rotation():
    u = torch.rand(3)
    q = torch.tensor([
        torch.sqrt(1-u[0])*torch.sin(2*math.pi*u[1]),
        torch.sqrt(1-u[0])*torch.cos(2*math.pi*u[1]),
        torch.sqrt(u[0])*torch.sin(2*math.pi*u[2]),
        torch.sqrt(u[0])*torch.cos(2*math.pi*u[2]),
    ])
    q = q / q.norm()
    w,x,y,z = q
    R = torch.tensor([
        [1-2*(y*y+z*z),  2*(x*y - z*w),  2*(x*z + y*w)],
        [2*(x*y + z*w),  1-2*(x*x+z*z),  2*(y*z - x*w)],
        [2*(x*z - y*w),  2*(y*z + x*w),  1-2*(x*x+y*y)]
    ], dtype=torch.float32)
    return R

t = torch.zeros((n_ligand,1), device=device)
mu_coord, gamma_coord = model.continuous_var_bayesian_update(t, sigma1=model.sigma1_coord, x=ligand_pos)
theta = torch.ones((n_ligand, K), device=device) / K

coord_pred, p0_h, k_hat = model.interdependency_modeling(
    time=t, protein_pos=protein_pos, protein_v=protein_v, batch_protein=batch_protein,
    theta_h_t=theta, mu_pos_t=mu_coord, batch_ligand=batch_ligand, gamma_coord=gamma_coord
)
print("coord_pred", coord_pred.shape)   # expect [n_ligand,3]
print("p0_h", p0_h.shape)               # expect [n_ligand,K]
print("k_hat", k_hat.shape)             # expect [n_ligand,1]

R = random_rotation()
protein_pos_rot = protein_pos @ R.T

mu_pos_rot = mu_coord @ R.T

coord_pred0, p0_0, q0 = model.interdependency_modeling(
    time=t, protein_pos=protein_pos, protein_v=protein_v, batch_protein=batch_protein,
    theta_h_t=theta, mu_pos_t=mu_coord, batch_ligand=batch_ligand, gamma_coord=gamma_coord
)
coord_pred1, p0_1, q1 = model.interdependency_modeling(
    time=t, protein_pos=protein_pos_rot, protein_v=protein_v, batch_protein=batch_protein,
    theta_h_t=theta, mu_pos_t=mu_pos_rot, batch_ligand=batch_ligand, gamma_coord=gamma_coord
)
print("max abs diff charges:", (q0 - q1).abs().max().item())


coord_pred torch.Size([6, 3])
p0_h torch.Size([6, 5])
k_hat torch.Size([6, 1])
max abs diff charges: 2.9802322387695312e-08
